In [72]:
import psycopg2
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

user = os.getenv("user")
password = os.getenv("password")
port = os.getenv("port")
host = "localhost"
database= "crypto_analytics"

try:
    conexao = psycopg2.connect(
        host = host,
        database = database,
        user = user,
        password = password,
        port = port
    )
    print("Conectado com sucesso!")
except Exception as e:
    print(repr(e))

Conectado com sucesso!


In [73]:
query_max_id = """
SELECT COALESCE(MAX(id_raw),0)
FROM crypto_silver
"""

with conexao.cursor() as cursor:
    
    cursor.execute(query_max_id)

    ultimo_id_processado = cursor.fetchone()[0]

print(ultimo_id_processado)

9


In [74]:
query = f"""SELECT * FROM crypto_raw
    WHERE id > {ultimo_id_processado};"""

with conexao.cursor() as cursor:
    cursor.execute(query)

    dados_raw = cursor.fetchall()

    colunas = [desc[0] for desc in cursor.description]

conexao.close()

df_raw = pd.DataFrame(dados_raw, columns=colunas)

display(df_raw)

,id,data_coleta,payload
0,10,2026-10-10 23:17:32.467008,"{'solana': {'brl': 548.31, 'eur': 98.14, 'usd'..."
1,11,2026-10-10 23:31:30.546213,"{'solana': {'brl': 548.26, 'eur': 98.13, 'usd'..."


In [75]:
registros = []

for _, linha in df_raw.iterrows():

    id_raw = linha["id"]

    payload = linha["payload"]
    data_coleta = linha["data_coleta"]

    for coin, info in payload.items():
        registros.append({
            "id_raw": id_raw,
            "coin": coin,
            "usd": info["usd"],
            "usd_market_cap": info["usd_market_cap"],
            "eur": info["eur"],
            "eur_market_cap": info["eur_market_cap"],
            "brl": info["brl"],
            "brl_market_cap": info["brl_market_cap"],
            "data_coleta": data_coleta
        })

df_silver = pd.DataFrame(registros)

display(df_silver)

,id_raw,coin,usd,usd_market_cap,eur,eur_market_cap,brl,brl_market_cap,data_coleta
0,10,solana,109.940000,6.476390e+10,98.140000,5.780942e+10,548.310000,3.229938e+11,2026-10-10 23:17:32.467008
1,10,bitcoin,82978.000000,1.667555e+12,74067.000000,1.488489e+12,413830.000000,8.316512e+12,2026-10-10 23:17:32.467008
2,10,cardano,0.248715,9.337710e+09,0.222008,8.335008e+09,1.240000,4.656949e+10,2026-10-10 23:17:32.467008
3,10,ripples,0.001972,1.676277e+05,0.001760,1.496275e+05,0.009835,8.360010e+05,2026-10-10 23:17:32.467008
4,10,ethereum,2507.410000,3.062155e+11,2238.160000,2.733335e+11,12505.080000,1.527173e+12,2026-10-10 23:17:32.467008
5,10,binancecoin,747.870000,9.958521e+10,667.560000,8.889155e+10,3729.830000,4.966563e+11,2026-10-10 23:17:32.467008
6,11,solana,109.930000,6.475283e+10,98.130000,5.779954e+10,548.260000,3.229385e+11,2026-10-10 23:31:30.546213
7,11,bitcoin,82971.000000,1.667522e+12,74062.000000,1.488460e+12,413798.000000,8.316350e+12,2026-10-10 23:31:30.546213
8,11,cardano,0.248941,9.346799e+09,0.222209,8.343121e+09,1.240000,4.661482e+10,2026-10-10 23:31:30.546213
9,11,ripples,0.001972,1.676277e+05,0.001760,1.496275e+05,0.009835,8.360010e+05,2026-10-10 23:31:30.546213


In [76]:
tickers = {
    "Bitcoin":"BTC",
    "Ethereum":"ETH",
    "Solana":"SOL",
    "Ripples":"XRP",
    "Cardano":"ADA",
    "Binancecoin":"BNB"
}

df_silver["coin"] = df_silver["coin"].str.title()
df_silver["tickers"] = df_silver["coin"].map(tickers)
df_silver.rename(columns={"index":"id_raw"})

display(df_silver)

,id_raw,coin,usd,usd_market_cap,eur,eur_market_cap,brl,brl_market_cap,data_coleta,tickers
0,10,Solana,109.940000,6.476390e+10,98.140000,5.780942e+10,548.310000,3.229938e+11,2026-10-10 23:17:32.467008,SOL
1,10,Bitcoin,82978.000000,1.667555e+12,74067.000000,1.488489e+12,413830.000000,8.316512e+12,2026-10-10 23:17:32.467008,BTC
2,10,Cardano,0.248715,9.337710e+09,0.222008,8.335008e+09,1.240000,4.656949e+10,2026-10-10 23:17:32.467008,ADA
3,10,Ripples,0.001972,1.676277e+05,0.001760,1.496275e+05,0.009835,8.360010e+05,2026-10-10 23:17:32.467008,XRP
4,10,Ethereum,2507.410000,3.062155e+11,2238.160000,2.733335e+11,12505.080000,1.527173e+12,2026-10-10 23:17:32.467008,ETH
5,10,Binancecoin,747.870000,9.958521e+10,667.560000,8.889155e+10,3729.830000,4.966563e+11,2026-10-10 23:17:32.467008,BNB
6,11,Solana,109.930000,6.475283e+10,98.130000,5.779954e+10,548.260000,3.229385e+11,2026-10-10 23:31:30.546213,SOL
7,11,Bitcoin,82971.000000,1.667522e+12,74062.000000,1.488460e+12,413798.000000,8.316350e+12,2026-10-10 23:31:30.546213,BTC
8,11,Cardano,0.248941,9.346799e+09,0.222209,8.343121e+09,1.240000,4.661482e+10,2026-10-10 23:31:30.546213,ADA
9,11,Ripples,0.001972,1.676277e+05,0.001760,1.496275e+05,0.009835,8.360010e+05,2026-10-10 23:31:30.546213,XRP


In [77]:
conexao = psycopg2.connect(
    host= host,
    database=  database,
    user= user,
    password= password,
    port= port
)

query_insert = """
INSERT INTO crypto_silver (
    id_raw,
    coin,
    ticker,
    usd,
    usd_market_cap,
    eur,
    eur_market_cap,
    brl,
    brl_market_cap,
    data_coleta)
VALUES (
    %s,%s,%s,%s,%s,%s,%s,%s,%s,%s
)"""

with conexao.cursor() as cursor:

    for _, linha in df_silver.iterrows():

        cursor.execute(
            query_insert,
            (
                linha["id_raw"],
                linha["coin"],
                linha["tickers"],
                linha["usd"],
                linha["usd_market_cap"],
                linha["eur"],
                linha["eur_market_cap"],
                linha["brl"],
                linha["brl_market_cap"],
                linha["data_coleta"]
            )
        )
    conexao.commit()
conexao.close()

print("Carga concluída")

Carga concluída
